# Supertonic Emotion Calibration

This Colab notebook extracts matched neutral, angry, and surprised styles, then creates reusable emotion difference JSON files. Use a GPU runtime, preferably a T4 or better.

Only use recordings you have permission to process. The notebook does not train one style from mixed emotions: it creates one full style per recording and then computes emotion deltas.

## References

- [supertonic.embed](https://github.com/kdrkdrkdr/supertonic.embed) for style extraction.
- [supertonic3-voice-clone](https://github.com/saurabhv749/supertonic3-voice-clone) for Supertonic 3 style optimization.
- [EmoShift](https://arxiv.org/abs/2601.22873) for emotion steering vectors.
- [RAVDESS](https://zenodo.org/records/1188976) for matched emotion recordings.

## Style calibration rules

Extract neutral, angry, and surprised as separate styles. A single speaker is enough for a voice-specific test, but a general emotion vector should use multiple speakers and subtract each speaker's neutral style first. Runtime steering targets `style_ttl` by default, normalizes TTL rows after blending, and leaves duration neutral unless explicitly enabled.

In [1]:
import os
import subprocess
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime in Runtime > Change runtime type before continuing.')
print(torch.cuda.get_device_name(0))
print(f'CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

RuntimeError: Enable a GPU runtime in Runtime > Change runtime type before continuing.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                 'torch', 'torchaudio', 'transformers', 'onnx',
                 'onnx2torch', 'onnxslim', 'librosa', 'soundfile',
                 'PyYAML', 'huggingface_hub'], check=True)
subprocess.run(['git', 'clone', '--depth', '1',
                 'https://github.com/kdrkdrkdr/supertonic.embed.git',
                 '/content/supertonic.embed'], check=False)
subprocess.run(['git', 'clone', '--depth', '1',
                 'https://github.com/supertone-inc/supertonic.git',
                 '/content/supertonic'], check=False)

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='Supertone/supertonic-3',
    local_dir='/content/supertonic-assets',
    allow_patterns=['onnx/*', 'voice_styles/M1.json'],
)
print('Downloaded model assets.')

## Upload matched recordings and resume safely

The next cell mounts Google Drive and stores recordings, the Hugging Face model cache, extracted styles, and an extraction manifest under `MyDrive/supertonic-emotion-calibration/`. If Colab disconnects, reconnect to the same Drive, rerun setup, and rerun the extraction cell. Completed emotion files are skipped automatically; only the missing emotion resumes.

Upload three files named `neutral.wav`, `angry.wav`, and `surprised.wav`. They should be the same speaker reading the same text or closely matched texts. For a general vector, repeat this notebook for multiple speakers and average the per-speaker deltas.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os
import shutil

# Persist all inputs and outputs outside the temporary Colab VM.
drive.mount('/content/drive')
checkpoint_root = Path('/content/drive/MyDrive/supertonic-emotion-calibration')
recording_dir = checkpoint_root / 'recordings'
recording_dir.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(checkpoint_root / 'huggingface-cache')

required = {'neutral.wav', 'angry.wav', 'surprised.wav'}
missing = required - {path.name for path in recording_dir.glob('*.wav')}
if missing:
    uploaded = files.upload()
    missing_uploads = missing - set(uploaded)
    if missing_uploads:
        raise ValueError(f'Missing recordings: {sorted(missing_uploads)}')
    for name in missing:
        shutil.copy2(name, recording_dir / name)

print(f'Persistent workspace: {checkpoint_root}')
print(f'Recordings ready: {sorted(path.name for path in recording_dir.glob("*.wav"))}')

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import subprocess
import sys
import yaml

extractor = Path('/content/supertonic.embed')
config_path = extractor / 'src/config.yaml'
config = yaml.safe_load(config_path.read_text())
config['model']['onnx_dir'] = '/content/supertonic-assets/onnx'
config['model']['presets'] = '/content/supertonic-assets/voice_styles'
config['timbre']['batch'] = 1
config_path.write_text(yaml.safe_dump(config, sort_keys=False))

output_dir = checkpoint_root / 'extracted-styles'
output_dir.mkdir(parents=True, exist_ok=True)
manifest_path = checkpoint_root / 'extraction_manifest.json'
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

for emotion in ('neutral', 'angry', 'surprised'):
    output_path = output_dir / f'{emotion}.json'
    if output_path.exists():
        manifest[emotion] = {'status': 'complete', 'path': str(output_path)}
        print(f'[skip] {emotion}: checkpoint already exists')
        continue

    wavlist = checkpoint_root / f'{emotion}_wavlist.txt'
    wavlist.write_text(f'{recording_dir / (emotion + ".wav")}|{emotion}\n')
    manifest[emotion] = {
        'status': 'running',
        'started_at': datetime.now(timezone.utc).isoformat(),
    }
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f'[start] {emotion}')
    subprocess.run([
        sys.executable, 'src/run_extract_style_batch.py',
        str(wavlist), '--out', str(output_dir),
    ], cwd=extractor, check=True)
    if not output_path.exists():
        raise FileNotFoundError(f'Extractor completed without writing {output_path}')
    manifest[emotion] = {
        'status': 'complete',
        'path': str(output_path),
        'completed_at': datetime.now(timezone.utc).isoformat(),
    }
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f'[saved] {output_path}')

print(json.dumps(manifest, indent=2))

In [ ]:
import json
import numpy as np


def read_style(path):
    payload = json.loads(Path(path).read_text())
    return {name: np.asarray(payload[name]['data'], dtype=np.float32).reshape(payload[name]['dims'])
            for name in ('style_ttl', 'style_dp')}


neutral = read_style(output_dir / 'neutral.json')
emotion_dir = Path('/content/emotion-styles')
emotion_dir.mkdir(exist_ok=True)
for emotion in ('angry', 'surprised'):
    emotional = read_style(output_dir / f'{emotion}.json')
    payload = {
        'style_ttl': {'data': (emotional['style_ttl'] - neutral['style_ttl']).tolist(), 'dims': list(neutral['style_ttl'].shape), 'type': 'float32'},
        'style_dp': {'data': (emotional['style_dp'] - neutral['style_dp']).tolist(), 'dims': list(neutral['style_dp'].shape), 'type': 'float32'},
        'metadata': {'emotion': emotion, 'kind': 'style_difference', 'neutral_style': str(output_dir / 'neutral.json'), 'emotional_style': str(output_dir / f'{emotion}.json')}
    }
    (emotion_dir / f'{emotion}.json').write_text(json.dumps(payload))
print(sorted(emotion_dir.glob('*.json')))

## Test the controls

Copy the generated JSONs into `assets/emotion_styles/` in the repository. Runtime blending normalizes TTL rows and leaves duration neutral by default. Use duration blending only after separate prosody evaluation.

In [ ]:
from google.colab import files

for path in sorted(emotion_dir.glob('*.json')):
    files.download(str(path))